# COCO-small MVP Pipeline (Colab)

???? ??????? ????????? ????????? `mvp_pipeline_colab.ipynb`,
?? ??? ???????? COCO 2017 ? ??????? ?? ????? ???????? (`area <= 32^2`).


In [ ]:
# Step 0: install dependencies in Colab runtime.
# ???? ?????-?? ?????? ??? ???????????, pip ?????? ????????? ??.
%pip install -q ultralytics pycocotools albumentations pyyaml tqdm


In [ ]:
# Step 1: mount Google Drive and define dataset paths.
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

DRIVE_ROOT = Path('/content/drive/MyDrive')
COCO_ZIPS_DIR = DRIVE_ROOT / 'datasets' / 'coco2017_zips'
COCO_RAW_DIR = DRIVE_ROOT / 'datasets' / 'coco2017_raw'
COCO_SMALL_YOLO_DIR = DRIVE_ROOT / 'datasets' / 'coco_small_yolo'

COCO_ZIPS_DIR.mkdir(parents=True, exist_ok=True)
COCO_RAW_DIR.mkdir(parents=True, exist_ok=True)
COCO_SMALL_YOLO_DIR.mkdir(parents=True, exist_ok=True)

print('COCO zips dir:', COCO_ZIPS_DIR)
print('COCO raw dir:', COCO_RAW_DIR)
print('COCO small yolo dir:', COCO_SMALL_YOLO_DIR)


In [ ]:
# Step 2: clone/open project repository in runtime.
from pathlib import Path
import subprocess
import sys

PROJECT_ROOT = Path('/content/small_objects_auto_aug')
REPO_URL = 'https://github.com/s44w/small_objects_auto_aug.git'

if not PROJECT_ROOT.exists():
    subprocess.check_call(['git', 'clone', REPO_URL, str(PROJECT_ROOT)])

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Using PROJECT_ROOT:', PROJECT_ROOT)


In [ ]:
# Step 3: download official COCO 2017 archives directly to Google Drive.
# ??? ?????? ???????? ?????? "?????????? ???????? ? Drive".
from src.data.coco_small_manager import download_coco_2017_archives

archive_paths = download_coco_2017_archives(COCO_ZIPS_DIR)
for name, path in archive_paths.items():
    print(f'[OK] {name}: {path}')


In [ ]:
# Step 4: extract COCO archives from Drive into Drive/raw folder.
# ??? ?????? ???????? ?????? "?????????? ???????? ? Drive" (????????????? raw).
from src.data.coco_small_manager import extract_coco_2017_archives

extracted = extract_coco_2017_archives(COCO_ZIPS_DIR, COCO_RAW_DIR, overwrite=False)
for name, path in extracted.items():
    print(f'[OK] extracted {name}: {path}')


In [ ]:
# Step 5: convert COCO raw -> YOLO (small objects only).
from src.data.coco_small_manager import CocoSmallPrepareConfig, prepare_coco_small_yolo

prepare_cfg = CocoSmallPrepareConfig(
    small_max_area=32.0**2,
    keep_images_without_small=False,
    include_crowd=False,
    splits=('train', 'val'),
    link_images=True,  # ???????? ????? ?? ???? ????????? ?? raw ???????????
)

prepare_report = prepare_coco_small_yolo(
    raw_coco_root=COCO_RAW_DIR,
    output_root=COCO_SMALL_YOLO_DIR,
    config=prepare_cfg,
)

print('is_valid:', prepare_report.is_valid)
print('num_classes:', len(prepare_report.class_names))
print('train stats:', prepare_report.splits.get('train', {}))
print('val stats:', prepare_report.splits.get('val', {}))


In [ ]:
# Step 6: build runtime config for pipeline (based on configs/coco_small_config.yaml).
import yaml

base_cfg_path = PROJECT_ROOT / 'configs' / 'coco_small_config.yaml'
runtime_cfg_path = PROJECT_ROOT / 'artifacts' / 'coco_small_runtime_config.yaml'
runtime_cfg_path.parent.mkdir(parents=True, exist_ok=True)

with base_cfg_path.open('r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

cfg['dataset']['mode'] = 'manual'
cfg['dataset']['root'] = str(COCO_SMALL_YOLO_DIR)
cfg['dataset']['raw_root'] = str(COCO_RAW_DIR)
cfg['training']['data_yaml'] = str(COCO_SMALL_YOLO_DIR / 'coco_small.yaml')

with runtime_cfg_path.open('w', encoding='utf-8') as f:
    yaml.safe_dump(cfg, f, allow_unicode=True, sort_keys=False)

print('Runtime config:', runtime_cfg_path)


In [ ]:
# Step 7: smoke test (subset -> analyze -> policy).
from src.data.subset_builder import build_yolo_subset
from src.analysis.dataset_analyzer import DatasetAnalyzerConfig, analyze_dataset
from src.policy.rule_engine import RuleEngineConfig, generate_policy_from_stats, save_policy_artifacts
from src.utils.io import load_yaml

RUN_SMOKE = True

if RUN_SMOKE:
    cfg = load_yaml(runtime_cfg_path)

    subset_root = PROJECT_ROOT / 'datasets' / 'coco_small_smoke'
    smoke_stats_dir = PROJECT_ROOT / 'artifacts' / 'coco_small_smoke' / 'stats'
    smoke_policy_dir = PROJECT_ROOT / 'artifacts' / 'coco_small_smoke' / 'policy'

    subset_report = build_yolo_subset(
        dataset_root=COCO_SMALL_YOLO_DIR,
        output_root=subset_root,
        train_images=120,
        val_images=40,
        seed=42,
        clean_output=True,
    )
    print('subset report:', subset_report)

    stats = analyze_dataset(
        dataset_root=subset_root,
        output_dir=smoke_stats_dir,
        splits=('train',),
        config=DatasetAnalyzerConfig(
            small_max_area=float(cfg['analysis']['small_max_area']),
            medium_max_area=float(cfg['analysis']['medium_max_area']),
            tiny_max_area=float(cfg['analysis']['tiny_max_area']),
            image_extensions=tuple(cfg['dataset'].get('image_extensions', ['.jpg', '.jpeg', '.png', '.bmp'])),
            generate_plots=False,
        ),
    )

    policy, decision_report = generate_policy_from_stats(
        stats,
        cfg=RuleEngineConfig.from_project_config(cfg),
    )
    paths = save_policy_artifacts(policy, decision_report, output_dir=smoke_policy_dir)

    print('Smoke OK')
    print('stats:', smoke_stats_dir / 'dataset_stats.json')
    print('policy:', paths['policy_json'])
else:
    print('Smoke skipped')


In [ ]:
# Step 8: full pipeline (analysis + policy, optional training/eval).
from src.pipeline_mvp import run_mvp_pipeline

RUN_TRAINING = False
RUN_EVAL = False
TRAIN_PROFILE = 'fast'  # fast | final | balanced | quality | hour | max_quality

outputs = run_mvp_pipeline(
    project_config_path=runtime_cfg_path,
    run_training=RUN_TRAINING,
    run_eval=RUN_EVAL,
    train_profile=TRAIN_PROFILE,
)
print(outputs)


In [ ]:
# Step 9 (optional): full training + eval in one call.
# ????????? ?????? ????? ?????? ????? ?????????? ??????.
from src.pipeline_mvp import run_mvp_pipeline

RUN_FULL_TRAIN_EVAL = False
if RUN_FULL_TRAIN_EVAL:
    outputs = run_mvp_pipeline(
        project_config_path=runtime_cfg_path,
        run_training=True,
        run_eval=True,
        train_profile='hour',
        auto_predict_for_eval=True,
    )
    print(outputs)
else:
    print('Full train+eval skipped')
